# 04 — Customer Segmentation

Segment customers using rule-based Frequency-Monetary-Satisfaction (FMS) tiers. This approach uses interpretable business rules rather than clustering algorithms, producing segments that can be directly actioned by a marketing team.

In [ ]:
from pathlib import Path
import pandas as pd
from src.data.load import load_foodhub
from src.features.build import clean_data, engineer_features
from src.analysis.segments import (
    compute_customer_metrics,
    create_segments,
    profile_segments,
)
from src.visualization.plots import (
    plot_segment_sizes,
    plot_segment_profiles,
    plot_segment_comparison,
)

In [ ]:
df = load_foodhub(Path("data/raw/foodhub_order.csv"))
df = clean_data(df)
df = engineer_features(df)

## Customer-Level Metrics

In [ ]:
metrics = compute_customer_metrics(df)
print(f"Unique customers: {len(metrics)}")
metrics.describe()

In [ ]:
print(f"Repeat customers: {(metrics['order_count'] > 1).sum()} ({(metrics['order_count'] > 1).mean():.1%})")
print(f"One-time customers: {(metrics['order_count'] == 1).sum()} ({(metrics['order_count'] == 1).mean():.1%})")

**65% of customers ordered only once.** This is the dominant pattern and a key business challenge — retention, not acquisition, is where the leverage is.

## Segmentation

In [ ]:
segmented = create_segments(metrics)
segmented[['customer_id', 'order_count', 'avg_spend', 'avg_rating',
           'frequency_tier', 'monetary_tier', 'satisfaction_tier', 'segment']].head(10)

## Segment Distribution

In [ ]:
segment_counts = segmented['segment'].value_counts()
print(segment_counts)
print(f'\nPercentages:\n{(segment_counts / len(segmented) * 100).round(1)}')

fig = plot_segment_sizes(segment_counts)
fig

## Segment Profiles

In [ ]:
profiles = profile_segments(segmented)
profiles

In [ ]:
fig = plot_segment_profiles(profiles, ["avg_orders", "avg_spend", "avg_prep_time", "avg_delivery_time"])
fig

## Segment Comparisons

In [ ]:
fig = plot_segment_comparison(segmented, "segment", "avg_spend")
fig

## Business Recommendations

| Segment | Action |
|---|---|
| **One-Time** | Re-engagement campaigns: first-order discount, personalized cuisine recommendations based on their single order |
| **Loyal High-Spender** | VIP treatment: priority delivery, exclusive restaurant access, loyalty rewards |
| **Regular** | Upsell: suggest higher-value items, bundle deals to increase average order value |
| **At-Risk** | Investigate: what's driving low satisfaction? Target with surveys and service recovery offers |